In [1]:
from pennylane import numpy as np
import pennylane as qml
import itertools
from itertools import combinations
from scipy.optimize import minimize
from math import comb
import sys
sys.path.append("../")
import qaoa_aux 
# import importlib
# importlib.reload(qaoa_aux)

In [2]:
def U_B(beta,edge_list):
    for edge_idx, (u, v) in enumerate(edge_list):
        theta = beta[edge_idx]
        qml.S(wires=u)
        qml.SingleExcitationMinus(-2 * theta,wires=[u, v])
        qml.adjoint(qml.S)(wires=u)

In [3]:
def U_C(gamma,cost_terms):
    # we have term index + cost terms so all the even numbered ZZ's
    for term_idx, (_, wires, coeff) in enumerate(cost_terms):
        theta = 2 * gamma[term_idx] * coeff
        # example for 4 consectuve z's
        #qml.PauliRot(theta, "ZZZZ", wires=[0,1,2,3])
        # pennylane will decompose this into CNOT+RZ ladder 
        qml.PauliRot(theta,"Z" * len(wires),wires=list(wires))

In [4]:
def U_C_MC(gamma,edge_list):
    edge_idx = 0
    for u, v in edge_list: 
        
        qml.CNOT(wires=(u, v))
        qml.RZ(gamma[edge_idx], wires=v)
        qml.CNOT(wires=(u, v))
        edge_idx += 1 

In [5]:
def objective(gammas,betas,circuit,hyperedge_list,cost_terms,weights):
    expectation_values = circuit(gammas, betas)
    value = 0.0

    # weighted constant term for each hyperedge
    for edge_idx, edge in enumerate(hyperedge_list):
        k = len(edge)
        #w_e * 1 - (1/2^k-1)
        value += weights[edge_idx] * (1 - 1 / (2 ** (k - 1)))

    # coeff is weighted in build_cost_terms()
    for term_idx, (_, _, coeff) in enumerate(cost_terms):
        value += coeff * expectation_values[term_idx]

    return value

In [6]:
def run_qaoa(
    hyperedge_list,
    n,
    weights,
    runtype="aon",
    num_layers=1,
    seed=42,
    scaling_constant=0.1,
    step_size=0.01,
    max_trials=100
):
    n_wires=n
    assert len(weights)==len(hyperedge_list)
    # Build mixer edges and cost terms
    cost_edge_list = qaoa_aux.clique_expansion(hyperedge_list)
    mixer_edge_list = qaoa_aux.connected_clique_expansion(hyperedge_list, n)
    cost_terms = qaoa_aux.build_cost_terms(hyperedge_list,weights)
    num_edges=len(cost_edge_list)

    # Balanced initial state
    target_weight = n_wires // 2
    initial_state = qaoa_aux.make_dicke_state(n_wires,target_weight)

    dev = qml.device("lightning.qubit",wires=n_wires)
    @qml.qnode(dev)
    def aon_circuit(gammas, betas):
        qml.StatePrep(initial_state,wires=range(n_wires))

        for layer in range(num_layers):
            U_C(gammas[:, layer],cost_terms)
            U_B(betas[:, layer],mixer_edge_list)

        observables = []

        for _, wires, _ in cost_terms:
            z_term = qml.prod(*[qml.PauliZ(wire)for wire in wires])
            observables.append(qml.expval(z_term))
        return observables

    
    @qml.qnode(dev)
    def mc_circuit(gammas,betas):
        qml.StatePrep(initial_state,wires=range(n_wires))
        for layer in range(num_layers):
            U_C_MC(gammas[:,layer],cost_edge_list)
            U_B(betas[:, layer],mixer_edge_list)
        observables = []

        for _, wires, _ in cost_terms:
            z_term = qml.prod(*[qml.PauliZ(wire)for wire in wires])
            observables.append(qml.expval(z_term))
        return observables
        
    # Select the QAOA circuit
    if runtype == "aon":
        circuit = aon_circuit
        n_gamma = len(cost_terms)
    elif runtype == "mc":
        circuit = mc_circuit
        n_gamma = len(cost_edge_list)
    else:
        raise ValueError("runtype must be 'aon' or 'mc'")
    
    # Random parameter initialization
    rng = np.random.default_rng(seed)
    
    gamma_params = np.array(scaling_constant * rng.random((n_gamma, num_layers)),requires_grad=True)
    beta_params = np.array(scaling_constant * rng.random((len(mixer_edge_list), num_layers)),requires_grad=True)
    
    opt = qml.AdamOptimizer(stepsize=step_size)
    objective_values = []
    
    # Define optimization objective
    def loss(g, b):
        return -objective(
            g,
            b,
            circuit,
            hyperedge_list,
            cost_terms,
            weights
        )
        
    initial_value = loss(gamma_params, beta_params)
    objective_values.append(float(initial_value))
    
    print(f"Initial objective = {-initial_value:.6f} | Runtype = {runtype}")

    # Optimization
    for trial in range(max_trials):
    
        gamma_params, beta_params = opt.step(loss,gamma_params,beta_params)
    
        current_value = loss(gamma_params, beta_params)
        objective_values.append(-float(current_value))
    
        if trial % 50 == 0:
            print(
                f"Iteration {trial:3d} | "
                f"objective = {-current_value:.6f}"
            )
    
    return {
        "gammas": gamma_params,
        "betas": beta_params,
        "objective_values": objective_values,
        "circuit": circuit,
        "cost_edge_list": cost_edge_list,
        "mixer_edge_list": mixer_edge_list,
        "cost_terms": cost_terms,
        "n_wires": n_wires
    }

In [29]:
P = np.load("../../data/poisson_16_12_3/P/03.npy")
w= np.load("../../data/poisson_16_12_3/w/03.npy")

hyperedge_list = qaoa_aux.matrix_to_hyperedges(P)

In [30]:

result = run_qaoa(
    hyperedge_list,
    weights=w,
    runtype="mc",
    n=12,
    num_layers=1,
    seed=42,
    scaling_constant=0.1,
    step_size=0.05,
    max_trials=300
)

Initial objective = 12.834087 | Runtype = mc
Iteration   0 | objective = 13.133693
Iteration  50 | objective = 15.016652
Iteration 100 | objective = 15.149308
Iteration 150 | objective = 15.182449
Iteration 200 | objective = 15.264121
Iteration 250 | objective = 15.265761


In [8]:
# import os
# import glob
# import pandas as pd
# folder_P = "../data/poisson_16_12_3/P"
# folder_w = "../data/poisson_16_12_3/w"
# p_files = sorted(glob.glob(f"{folder_P}/*.npy"))[:20]

# results_max_bisection_2 = []

# for p_file in p_files:

#     instance = os.path.splitext(os.path.basename(p_file))[0]
#     P = np.load(p_file)
#     w_file = os.path.join(folder_w,f"{instance}.npy")
#     w = np.load(w_file)
#     hyperedge_list = qaoa_aux.matrix_to_hyperedges(P)
#     n = P.shape[0]

#     #circuit depth
#     for p in [3]:

#         print(f"\nRunning instance {instance} | p={p}")

#         result = run_qaoa_modified(hyperedge_list,weights=w,n=n,num_layers=p,seed=42,scaling_constant=5e-2,step_size=0.01,max_trials=200)

#         final_objective = float(result["objective_values"][-1])

#         initial_objective = float(result["objective_values"][0])

#         results_max_bisection_2.append({
#             "dataset": "poisson_16_12_3",
#             "objective":"max_bisection",
#             "instance": instance,
#             "p": p,
#             "n_vertices": n,
#             "n_hyperedges": len(hyperedge_list),
#             "initial_objective": initial_objective,
#             "final_objective": final_objective,

#             "gammas": np.array2string(np.asarray(result["gammas"]),separator=","),
#             "betas": np.array2string(np.asarray(result["betas"]),separator=",")
#         })


In [16]:
import pandas as pd
df = pd.DataFrame(results_max_bisection)
csv_path = "poisson_16_12_3_qaoa_max_bisection.csv"

df.to_csv(csv_path,index=False)

print(f"saved results to: {csv_path}")

saved results to: poisson_16_12_3_qaoa_max_bisection.csv


In [23]:
# def run_qaoa_modified(
#     hyperedge_list,
#     n,
#     weights,
#     num_layers=1,
#     seed=42,
#     scaling_constant=0.1,
#     step_size=0.01,
#     max_trials=100
# ):
#     n_wires=n
#     assert len(weights)==len(hyperedge_list)
#     # Build mixer edges and cost terms
#     edge_list = qaoa_aux.connected_clique_expansion(hyperedge_list,n)
#     cost_terms = qaoa_aux.build_cost_terms(hyperedge_list,weights)

#     # Balanced initial state
#     target_weight = n_wires // 2
#     initial_state = qaoa_aux.make_dicke_state(n_wires,target_weight)

#     dev = qml.device("lightning.qubit",wires=n_wires)
#     @qml.qnode(dev)
#     def circuit(gammas, betas):
#         qml.StatePrep(initial_state,wires=range(n_wires))

#         for layer in range(num_layers):
#             U_C(gammas[:, layer],cost_terms)
#             U_B(betas[:, layer],edge_list)

#         observables = []

#         for _, wires, _ in cost_terms:
#             z_term = qml.prod(*[qml.PauliZ(wire)for wire in wires])
#             observables.append(qml.expval(z_term))
#         return observables

#     # Random parameter initialization
#     rng = np.random.default_rng(seed)
#     beta_params = np.array(scaling_constant * rng.random((len(edge_list), num_layers)),requires_grad=True)
#     gamma_params = np.array(scaling_constant * rng.random((len(cost_terms), num_layers)),requires_grad=True)

#     opt = qml.AdamOptimizer(stepsize=step_size)
#     objective_values = []
#     initial_value = objective(gamma_params,beta_params,circuit,hyperedge_list,cost_terms,weights)
#     objective_values.append(initial_value)

#     print(f"Initial cut = {initial_value:.6f}")

#     # Optimization
#     for trial in range(max_trials):

#         gamma_params, beta_params = opt.step(
#             lambda g, b: -objective(
#                 g,
#                 b,
#                 circuit,
#                 hyperedge_list,
#                 cost_terms,
#                 weights
#             ),
#             gamma_params,
#             beta_params
#         )

#         current_value = objective(gamma_params,beta_params,circuit,hyperedge_list,cost_terms,weights)
#         objective_values.append(current_value)

#         if trial % 10 == 0:
#             print(f"Iteration {trial:3d} | "f"cut = {current_value:.6f}")

#     return {
#         "gammas": gamma_params,
#         "betas": beta_params,
#         "objective_values": objective_values,
#         "circuit": circuit,
#         "edge_list": edge_list,
#         "cost_terms": cost_terms,
#         "n_wires": n_wires
#     }

In [33]:
import os
import glob
import pandas as pd

def get_edge_list(P):
    edges=[]
    for idx in range(P.shape[1]):
        #indices that are not 0
        edge = tuple(np.flatnonzero(P[:,idx]))
        edges.append(edge)
    return edges
    
folder = "/Users/stevehalley/Code/Github/QuantumHypergraphPartitioning/data/congress-bills/numpy_20"
p_files = sorted(glob.glob(f"{folder}/*.npy"))

results_max_bisection_congress = []



for p_file in p_files:
    
    P = np.load(p_file)
    instance = os.path.splitext(os.path.basename(p_file))[0]

    edges = get_edge_list(P)
    w = np.ones(len(edges))
    hyperedge_list = qaoa_aux.matrix_to_hyperedges(P)
    n = P.shape[0]

    

    ###temp

    #circuit depth
    for p in [3]:
        

        print(f"\nRunning instance {instance} | p={p}")
        if n>16:
            result = run_qaoa(hyperedge_list,runtype="mc",weights=w,n=n,num_layers=p,seed=42,scaling_constant=5e-2,step_size=0.01,max_trials=300)
        else:
            result = run_qaoa(hyperedge_list,runtype="aon",weights=w,n=n,num_layers=p,seed=42,scaling_constant=5e-2,step_size=0.01,max_trials=300)

        final_objective = float(result["objective_values"][-1])

        initial_objective = float(result["objective_values"][0])

        results_max_bisection_congress.append({
            "dataset": "congress",
            "objective":"max_bisection",
            "instance": instance,
            "p": p,
            "n_vertices": n,
            "n_hyperedges": len(hyperedge_list),
            "initial_objective": initial_objective,
            "final_objective": final_objective,

            "gammas": np.array2string(np.asarray(result["gammas"]),separator=","),
            "betas": np.array2string(np.asarray(result["betas"]),separator=",")
        })
    



Running instance congress-bills_01 | p=3
Initial objective = 1.000000 | Runtype = aon
Iteration   0 | objective = 1.000000
Iteration  50 | objective = 1.000000
Iteration 100 | objective = 1.000000
Iteration 150 | objective = 1.000000
Iteration 200 | objective = 1.000000
Iteration 250 | objective = 1.000000

Running instance congress-bills_02 | p=3
Initial objective = 1.322784 | Runtype = aon
Iteration   0 | objective = 1.329620
Iteration  50 | objective = 1.992372
Iteration 100 | objective = 1.999956
Iteration 150 | objective = 2.000000
Iteration 200 | objective = 2.000000
Iteration 250 | objective = 2.000000

Running instance congress-bills_03 | p=3
Initial objective = 1.782644 | Runtype = aon
Iteration   0 | objective = 1.791712
Iteration  50 | objective = 2.622534
Iteration 100 | objective = 2.911607
Iteration 150 | objective = 2.959397
Iteration 200 | objective = 2.976577
Iteration 250 | objective = 2.988017

Running instance congress-bills_04 | p=3
Initial objective = 3.850464 | 

IOStream.flush timed out
IOStream.flush timed out


Iteration 100 | objective = 22.882450


IOStream.flush timed out


Iteration 150 | objective = 23.260645
Iteration 200 | objective = 23.510504


IOStream.flush timed out
IOStream.flush timed out


Iteration 250 | objective = 23.679861

Running instance congress-bills_15 | p=3
Initial objective = 1.000000 | Runtype = aon
Iteration   0 | objective = 1.000000
Iteration  50 | objective = 1.000000
Iteration 100 | objective = 1.000000
Iteration 150 | objective = 1.000000
Iteration 200 | objective = 1.000000
Iteration 250 | objective = 1.000000

Running instance congress-bills_16 | p=3
Initial objective = 2.263499 | Runtype = aon
Iteration   0 | objective = 2.275630
Iteration  50 | objective = 3.226325
Iteration 100 | objective = 3.769762
Iteration 150 | objective = 3.903267
Iteration 200 | objective = 3.922313
Iteration 250 | objective = 3.947482

Running instance congress-bills_17 | p=3
Initial objective = 1.322784 | Runtype = aon
Iteration   0 | objective = 1.329620
Iteration  50 | objective = 1.992372
Iteration 100 | objective = 1.999956
Iteration 150 | objective = 2.000000
Iteration 200 | objective = 2.000000
Iteration 250 | objective = 2.000000

Running instance congress-bills_18

In [35]:
import pandas as pd
df = pd.DataFrame(results_max_bisection_congress)
csv_path = "congress_qaoa_max_bisection.csv"

df.to_csv(csv_path,index=False)

print(f"saved results to: {csv_path}")

saved results to: congress_qaoa_max_bisection.csv


In [8]:
import os
import glob
import pandas as pd
folder_P = "../data/poisson_16_12_4/P"
folder_w = "../data/poisson_16_12_4/w"
p_files = sorted(glob.glob(f"{folder_P}/*.npy"))[:20]

results_max_bisection_4 = []

for p_file in p_files:

    instance = os.path.splitext(os.path.basename(p_file))[0]
    P = np.load(p_file)
    w_file = os.path.join(folder_w,f"{instance}.npy")
    w = np.load(w_file)
    hyperedge_list = qaoa_aux.matrix_to_hyperedges(P)
    n = P.shape[0]

    #circuit depth
    for p in [3]:

        print(f"\nRunning instance {instance} | p={p}")

        result = run_qaoa(hyperedge_list,runtype="aon",weights=w,n=n,num_layers=p,seed=42,scaling_constant=5e-2,step_size=0.01,max_trials=300)

        final_objective = float(result["objective_values"][-1])

        initial_objective = float(result["objective_values"][0])

        results_max_bisection_4.append({
            "dataset": "poisson_16_12_4",
            "objective":"max_bisection",
            "cost_hamiltonian":"AoN",
            "instance": instance,
            "p": p,
            "n_vertices": n,
            "n_hyperedges": len(hyperedge_list),
            "initial_objective": initial_objective,
            "final_objective": final_objective,

            "gammas": np.array2string(np.asarray(result["gammas"]),separator=","),
            "betas": np.array2string(np.asarray(result["betas"]),separator=",")
        })



Running instance 01 | p=3
Initial objective = 14.105714 | Runtype = aon
Iteration   0 | objective = 14.216450
Iteration  50 | objective = 15.894248
Iteration 100 | objective = 15.974044
Iteration 150 | objective = 15.986324
Iteration 200 | objective = 15.991645
Iteration 250 | objective = 15.994839

Running instance 02 | p=3
Initial objective = 13.631507 | Runtype = aon
Iteration   0 | objective = 13.734212
Iteration  50 | objective = 15.780784
Iteration 100 | objective = 15.958627
Iteration 150 | objective = 15.981183
Iteration 200 | objective = 15.989696
Iteration 250 | objective = 15.993429

Running instance 03 | p=3
Initial objective = 13.336489 | Runtype = aon
Iteration   0 | objective = 13.430356
Iteration  50 | objective = 15.633542
Iteration 100 | objective = 15.923971
Iteration 150 | objective = 15.966722
Iteration 200 | objective = 15.978212
Iteration 250 | objective = 15.983593

Running instance 04 | p=3
Initial objective = 14.034882 | Runtype = aon
Iteration   0 | objectiv

In [10]:
import pandas as pd
df = pd.DataFrame(results_max_bisection_4)
csv_path = "poisson_16_12_4_qaoa_max_bisection.csv"

df.to_csv(csv_path,index=False)

print(f"saved results to: {csv_path}")

saved results to: poisson_16_12_4_qaoa_max_bisection.csv


In [11]:
import os
import glob
import pandas as pd
folder_P = "../data/poisson_16_12_5/P"
folder_w = "../data/poisson_16_12_5/w"
p_files = sorted(glob.glob(f"{folder_P}/*.npy"))[:20]

results_max_bisection_5 = []

for p_file in p_files:

    instance = os.path.splitext(os.path.basename(p_file))[0]
    P = np.load(p_file)
    w_file = os.path.join(folder_w,f"{instance}.npy")
    w = np.load(w_file)
    hyperedge_list = qaoa_aux.matrix_to_hyperedges(P)
    n = P.shape[0]

    #circuit depth
    for p in [3]:

        print(f"\nRunning instance {instance} | p={p}")

        result = run_qaoa(hyperedge_list,runtype="aon",weights=w,n=n,num_layers=p,seed=42,scaling_constant=5e-2,step_size=0.01,max_trials=300)

        final_objective = float(result["objective_values"][-1])

        initial_objective = float(result["objective_values"][0])

        results_max_bisection_4.append({
            "dataset": "poisson_16_12_5",
            "objective":"max_bisection",
            "cost_hamiltonian":"AoN",
            "instance": instance,
            "p": p,
            "n_vertices": n,
            "n_hyperedges": len(hyperedge_list),
            "initial_objective": initial_objective,
            "final_objective": final_objective,

            "gammas": np.array2string(np.asarray(result["gammas"]),separator=","),
            "betas": np.array2string(np.asarray(result["betas"]),separator=",")
        })



Running instance 01 | p=3
Initial objective = 14.730946 | Runtype = aon
Iteration   0 | objective = 14.813612
Iteration  50 | objective = 15.978135
Iteration 100 | objective = 15.995437
Iteration 150 | objective = 15.998029
Iteration 200 | objective = 15.998929
Iteration 250 | objective = 15.999349

Running instance 02 | p=3
Initial objective = 15.543316 | Runtype = aon
Iteration   0 | objective = 15.576534
Iteration  50 | objective = 15.996661
Iteration 100 | objective = 15.999819
Iteration 150 | objective = 15.999978
Iteration 200 | objective = 15.999986
Iteration 250 | objective = 15.999972

Running instance 03 | p=3
Initial objective = 14.913949 | Runtype = aon
Iteration   0 | objective = 14.980084
Iteration  50 | objective = 15.984047
Iteration 100 | objective = 15.997375
Iteration 150 | objective = 15.999010
Iteration 200 | objective = 15.999533
Iteration 250 | objective = 15.999760

Running instance 04 | p=3
Initial objective = 15.294995 | Runtype = aon
Iteration   0 | objectiv

IOStream.flush timed out



Running instance 10 | p=3
Initial objective = 15.049037 | Runtype = aon
Iteration   0 | objective = 15.138413
Iteration  50 | objective = 15.975005
Iteration 100 | objective = 15.996526
Iteration 150 | objective = 15.999177
Iteration 200 | objective = 15.999790
Iteration 250 | objective = 15.999948


In [13]:
import pandas as pd
df = pd.DataFrame(results_max_bisection_5)
csv_path = "poisson_16_12_5_qaoa_max_bisection.csv"

df.to_csv(csv_path,index=False)

print(f"saved results to: {csv_path}")

saved results to: poisson_16_12_5_qaoa_max_bisection.csv
